# Caminho dos Direitos — Anotação por voz com IA
O agente **fala** a anotação da visita (pelo microfone, aqui mesmo) ou digita. Este notebook:

1. **grava e transcreve** a fala aqui mesmo (Whisper) e apaga o áudio;
2. **remove os dados clínicos** (pressão, peso, altura, glicemia, nome de doença ou remédio) e avisa: *registre no e-SUS*;
3. **analisa e organiza** o texto filtrado em um **resumo da visita** (situação, prioridade, próximos passos, pendências) e em 10 marcações de direitos (regras + Claude, se houver chave);
4. **guarda só as marcações que o agente confirmar**: nem o texto, nem o áudio, nem o dado clínico.

**Como usar:** Ambiente de execução → Executar tudo. A gravação pelo microfone fica no fim (Teste 3): clique em *Permitir* quando o navegador pedir o microfone, fale e clique em **Parar e organizar**.
Sem chave de API funciona só com as regras. Para usar o Claude: ícone de chave (Secrets) → `ANTHROPIC_API_KEY` → ative para este notebook.

In [ ]:
!pip install -q anthropic faster-whisper

In [ ]:
# Compatibilidade: o faster-whisper passa "metadata_errors" para o PyAV, que removeu esse parâmetro.
import av
import av.container

_av_open_real = av.container.open   # sempre a função original do PyAV, mesmo rodando a célula de novo

def _av_open_compat(*args, **kwargs):
    kwargs.pop("metadata_errors", None)
    return _av_open_real(*args, **kwargs)

av.open = _av_open_compat

## Importações

In [ ]:
from __future__ import annotations

import json
import os
import re
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

## 1. As 10 marcações (o que o app guarda) e o que cada uma muda no caso

In [ ]:
CATEGORIAS = {
    "dificuldade_locomocao": {
        "rotulo": "Dificuldade de se locomover",
        "acao": "Atendimento em casa ou acompanhado",
        "formulario": {},
    },
    "dificuldade_ir_ao_posto": {
        "rotulo": "Dificuldade de chegar ao posto (distância, passagem, sem acompanhante)",
        "acao": "Encaminhamento por visita domiciliar ou atendimento acompanhado",
        "formulario": {},
    },
    "pessoa_com_deficiencia": {
        "rotulo": "Pessoa com deficiência",
        "acao": "Conferir BPC; incluir laudo na lista do que levar",
        "formulario": {"pcd": "sim"},
    },
    "precisa_atendimento_medico": {
        "rotulo": "Precisa de atendimento médico",
        "acao": "Encaminhar para a UBS (sem registrar o motivo aqui)",
        "formulario": {},
    },
    "uso_continuo_remedio_ou_fralda": {
        "rotulo": "Uso contínuo de remédio ou fralda",
        "acao": "Conferir Farmácia Popular",
        "formulario": {"precisa_remedio_ou_fralda": "sim"},
    },
    "precisa_acompanhante": {
        "rotulo": "Precisa de acompanhante",
        "acao": "Confirmar quem acompanha e se a pessoa está livre no dia e horário",
        "formulario": {},
    },
    "melhor_horario": {
        "rotulo": "Melhor horário",
        "acao": "Marcar atendimentos neste período",
        "formulario": {},
    },
    "problema_beneficio": {
        "rotulo": "Problema com benefício",
        "acao": "Verificar a situação do benefício e do CadÚnico no CRAS (ou INSS, se for BPC/aposentadoria)",
        "formulario": {},
    },
    "problema_documentos": {
        "rotulo": "Problema com documentos",
        "acao": "Orientar emissão ou 2ª via (Poupatempo / cartório)",
        "formulario": {},
    },
    "urgencia_social": {
        "rotulo": "Situação de urgência (comida, moradia, luz)",
        "acao": "Prioridade no caso e aviso ao coordenador",
        "formulario": {},
    },
}

# Padrões em texto sem acento e minúsculo (ver _simples). Cada um indica a marcação.
PADROES = {
    "dificuldade_locomocao": [
        r"acamad", r"nao (consegue|pode) (andar|sair|levantar|subir escada)", r"nao sai de casa",
        r"cadeira de rodas", r"cadeirante", r"andador", r"muleta", r"bengala", r"mobilidade",
        r"dificuldade (de|para|pra) (andar|se locomover|locomocao|sair)", r"anda com dificuldade",
        r"nao anda\b",
    ],
    "dificuldade_ir_ao_posto": [
        r"mora (muito )?longe", r"longe (do|da) (posto|ubs|cras)",
        r"sem dinheiro (pra|para|da) (passagem|conducao|onibus)",
        r"nao tem como ir", r"nao tem quem (leve|acompanhe)",
    ],
    "pessoa_com_deficiencia": [
        r"deficien", r"\bpcd\b", r"cadeirante", r"\bcego\b", r"\bcega\b", r"cegueira", r"\bsurd[oa]",
        r"autis", r"sindrome de down", r"paralisia", r"amputad",
    ],
    "precisa_atendimento_medico": [
        r"precisa (de |ir ao |ir no |passar no )?(medico|consulta|atendimento)", r"esta doente", r"passando mal",
        r"sentindo dor", r"com dor", r"febre", r"ferida", r"machucad", r"nao vai ao medico",
        r"sem consulta", r"nao (passa|passou) (no|pelo) medico",
    ],
    "uso_continuo_remedio_ou_fralda": [
        r"remedio (de uso )?continuo", r"remedio todo dia", r"toma remedio", r"fralda", r"medicacao",
        r"remedios? controlad",
    ],
    "precisa_acompanhante": [
        r"(precisa|tem que|so pode|so consegue) ir (com|junto|acompanhad)",
        r"precisa de (um |uma )?(acompanhante|alguem (pra|para) (levar|ir junto|acompanhar))",
        r"nao (pode|consegue) ir sozinh", r"\bacompanhante\b", r"nao tem quem (leve|acompanhe)",
    ],
    "melhor_horario": [
        r"(so|somente|apenas) (esta |fica |ta )?(disponivel|livre|pode) (a|de|pela|na|no periodo da) (manha|tarde|noite)",
        r"(melhor|prefere|so pode|so consegue)(\s+\w+){0,3}?\s+(a|de|pela|na) (manha|tarde|noite)",
        r"(disponivel|livre) (a|de|pela|na) (manha|tarde|noite)",
    ],
    "problema_beneficio": [
        r"(beneficio|bolsa familia|\bbpc\b|loas|aposentadoria|auxilio|pensao)\w*(\s+\w+){0,5}?"
        r"\s+(cortad|bloquead|suspens|cancelad|negad|atrasad|indeferid)",
        r"(cortaram|bloquearam|suspenderam|cancelaram|negaram) (o |a )?(beneficio|bolsa|bpc|loas|aposentadoria|auxilio|pensao)",
        r"(parou|deixou) de receber", r"nao (esta |ta )?receb(e|endo) (o |a )?(beneficio|bolsa|bpc|aposentadoria|auxilio|pensao)",
        r"problema (no|com o|com) (beneficio|bolsa|bpc|cadunico|cadastro)",
        r"(cadunico|cadastro unico|cadastro)(\s+\w+){0,2}?\s+(desatualizad|vencid|bloquead|pendente)",
        r"nao consegue (sacar|receber)", r"cartao (do bolsa familia )?bloquead",
        r"nunca (recebeu|pediu) (o |a )?(beneficio|bolsa|bpc)", r"benefici\w* (negad|cortad)",
    ],
    "problema_documentos": [
        r"sem (nenhum )?documento", r"sem (rg|cpf|certidao|identidade)",
        r"nao tem (nenhum )?(rg|cpf|certidao|documento|identidade|titulo)",
        r"(perdeu|roubaram|extraviou) (o |a |os |as |seus |suas )?(rg|cpf|certidao|documento|identidade|carteira)",
        r"documentos? (perdid|roubad|vencid|rasgad|extraviad|irregular)",
        r"(rg|cpf|identidade) (perdid|roubad|vencid|irregular|suspens|cancelad|pendente)",
        r"segunda via", r"2a via", r"nao foi registrad", r"sem registro de nascimento",
        r"problema (no|com o|com os|com) (documento|rg|cpf|certidao)",
    ],
    "urgencia_social": [
        r"sem comida", r"passando fome", r"nao tem (o )?que comer", r"geladeira vazia", r"despej",
        r"casa (caindo|rachada|condenada)", r"desabamento", r"alagamento", r"luz (foi |esta |ta )?cortad", r"cortaram (a|minha|nossa) luz",
        r"cortaram a luz", r"sem luz",
    ],
}
NEGACAO = re.compile(r"\b(nao|nem|nunca|nenhum|nenhuma)\b[^.;!?]{0,15}$")
LAUDO_SIM = re.compile(r"(tem|possui|com) laudo")
LAUDO_NAO = re.compile(r"(nao tem|sem|falta) laudo")

# Usados nas sugestões (só na tela, nada disso é salvo)
PERIODO = re.compile(r"manha|tarde|noite")
NOME_PERIODO = {"manha": "manhã", "tarde": "tarde", "noite": "noite"}
LOCAIS = {"poupatempo": "Poupatempo", "cras": "CRAS", "creas": "CREAS", "ubs": "UBS", "posto": "posto",
          "inss": "INSS", "cartorio": "cartório", "defensoria": "Defensoria", "caps": "CAPS"}
COMPROMISSO = re.compile(r"(marcad|agendad|agendament)")
LOCAL = re.compile(r"\b(" + "|".join(LOCAIS) + r")\b")
DATA = re.compile(r"dia \d{1,2}( de (janeiro|fevereiro|marco|abril|maio|junho|julho|agosto|setembro|outubro|"
                  r"novembro|dezembro))?|\b\d{1,2}/\d{1,2}\b")
ACOMPANHANTE = re.compile(r"\b(com|pelo|pela) (o |a )?(filh[oa]|net[oa]|marido|esposa|irma[o]?|vizinh[oa]|"
                          r"sobrinh[oa]|genro|nora|cuidador[a]?)\b")
PESSOA = re.compile(r"\b((?i:dona|seu|sr|sra))\.? [A-ZÀ-Ú][\wÀ-ú]+")

## 2. Dado clínico: detectado, removido e nunca guardado

In [ ]:
CLINICO = {
    "pressão": r"press[aã]o( arterial)?( d[eao])?\s*(alta|baixa|\d{1,3}\s*(x|/|por)\s*\d{1,3})|\b\d{1,2}\s*(x|por)\s*\d{1,2}\b",
    "peso": r"\b(pes(a|o|ando)|com)?\s*\d{2,3}([.,]\d+)?\s*(kg|quilos?)\b|\bpes(a|o|ando)\s+\d{2,3}\b",
    "altura": r"\b(mede|altura( de)?)?\s*([12][.,]\d{2}\s*(m|metros?)\b|1 metro e \d{2}\b|\d{3}\s*cm\b)",
    "glicemia": r"\b(glicemia|glicose|a[cç][uú]car no sangue)\D{0,12}\d{2,3}",
    "temperatura": r"\b3\d([.,]\d)?\s*(graus|°)",
    "doença ou condição": (
        r"\b(diabet|hipertens|c[aâ]ncer|tumor|hiv\b|aids\b|tuberculose|hansen[ií]ase|hepatite|avc\b|derrame|"
        r"infarto|card[ií]ac|epileps|depress|esquizofren|bipolar|alzheimer|dem[eê]ncia|parkinson|asma\b|"
        r"dpoc\b|insufici[eê]ncia renal|hemodi[aá]lise|s[ií]filis|cid\b|gravidez de risco)\w*"
    ),
    "nome de remédio": (
        r"\b(insulina|losartan|metformin|captopril|enalapril|sinvastatin|rivotril|clonazepam|fluoxetin|omeprazol)\w*"
    ),
}
# Alguns dados clínicos, mesmo removidos, indicam uma marcação de direitos (sem guardar o detalhe)
CLINICO_SUGERE = {"nome de remédio": "uso_continuo_remedio_ou_fralda"}
MARCADOR = "[dado de saúde: registrar no e-SUS]"


def _simples(txt: str) -> str:
    """minúsculo, sem acento (para os padrões funcionarem com qualquer transcrição)."""
    t = unicodedata.normalize("NFKD", txt.lower())
    return "".join(c for c in t if not unicodedata.combining(c))


def filtrar_clinico(texto: str) -> tuple[str, list[str]]:
    """Troca cada dado clínico pelo MARCADOR. Devolve (texto filtrado, tipos encontrados — sem os valores)."""
    achados = []
    for tipo, padrao in CLINICO.items():
        novo = re.sub(padrao, MARCADOR, texto, flags=re.IGNORECASE)
        if novo != texto:
            achados.append(tipo)
            texto = novo
    texto = re.sub(rf"({re.escape(MARCADOR)}[\s,;e]*)+", MARCADOR + " ", texto).strip()
    return texto, achados

## 3. Transcrição (Whisper local) e gravação pelo microfone

In [ ]:
_MODELO_WHISPER = None
# Palavras do dia a dia do agente: ajudam o Whisper a acertar siglas e nomes de programas.
VOCABULARIO = ("Anotação de visita domiciliar. CRAS, UBS, CadÚnico, Bolsa Família, BPC, Farmácia Popular, "
               "Tarifa Social, Pé-de-Meia, Poupatempo, laudo, e-SUS, acamada, cadeirante.")


def transcrever(caminho_audio: str | Path, tamanho: str = "small", apagar_audio: bool = True) -> str:
    """Áudio (ogg, mp3, m4a, wav, webm...) -> texto em português, rodando no próprio servidor.

    No Colab: !pip install -q faster-whisper   (usa GPU se houver; 'small' roda bem em CPU).
    O áudio é apagado depois da transcrição (apagar_audio=True): o app não guarda a voz da pessoa.
    """
    global _MODELO_WHISPER
    from faster_whisper import WhisperModel  # import aqui: só quem transcreve precisa instalar

    if _MODELO_WHISPER is None:
        try:
            import torch
            dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            dispositivo = "cpu"
        _MODELO_WHISPER = WhisperModel(tamanho, device=dispositivo,
                                       compute_type="float16" if dispositivo == "cuda" else "int8")
    try:
        trechos, info = _MODELO_WHISPER.transcribe(str(caminho_audio), language="pt", vad_filter=True,
                                                   initial_prompt=VOCABULARIO)
        texto = " ".join(t.text.strip() for t in trechos).strip()
        print(f"Áudio recebido: {info.duration:.1f} s · fala detectada: {info.duration_after_vad:.1f} s")
        if not texto and info.duration > 1:
            # o filtro de voz pode cortar fala baixa: tenta de novo sem ele
            print("Nenhuma fala detectada pelo filtro; tentando de novo sem o filtro de voz…")
            trechos, _ = _MODELO_WHISPER.transcribe(str(caminho_audio), language="pt", vad_filter=False,
                                                    initial_prompt=VOCABULARIO)
            texto = " ".join(t.text.strip() for t in trechos).strip()
        if not texto:
            print("⚠️  Transcrição vazia. Confira: o microfone certo está selecionado no navegador "
                  "(ícone de câmera/microfone na barra de endereço)? Fale perto e por alguns segundos.")
    finally:
        if apagar_audio:
            Path(caminho_audio).unlink(missing_ok=True)
    return texto


_JS_GRAVAR = """
async function gravarAnotacao(maxMs) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const partes = [];
  rec.ondataavailable = e => partes.push(e.data);
  const caixa = document.createElement('div');
  caixa.style = 'font-family:sans-serif;margin:8px 0;display:flex;gap:12px;align-items:center';
  const btn = document.createElement('button');
  btn.textContent = '⏹  Parar e organizar';
  btn.style = 'padding:10px 18px;font-size:15px;border-radius:8px;border:0;background:#c0392b;color:#fff;cursor:pointer';
  const info = document.createElement('span');
  const t0 = Date.now();
  const relogio = setInterval(() => {
    const s = Math.floor((Date.now() - t0) / 1000);
    info.textContent = '🔴 gravando… ' + Math.floor(s / 60) + ':' + String(s % 60).padStart(2, '0');
  }, 250);
  caixa.appendChild(btn); caixa.appendChild(info);
  document.querySelector('#output-area').appendChild(caixa);
  return await new Promise(resolve => {
    rec.onstop = () => {
      clearInterval(relogio); stream.getTracks().forEach(t => t.stop()); caixa.remove();
      const leitor = new FileReader();
      leitor.onloadend = () => resolve(leitor.result.split(',')[1]);
      leitor.readAsDataURL(new Blob(partes, {type: rec.mimeType}));
    };
    btn.onclick = () => rec.stop();
    rec.start();
    setTimeout(() => { if (rec.state === 'recording') rec.stop(); }, maxMs);
  });
}
"""


def gravar_do_microfone(max_segundos: int = 180, arquivo: str | Path = "anotacao.webm",
                        ouvir: bool = False) -> Path:
    """Grava pelo microfone do navegador (Colab) até o agente clicar em 'Parar' ou acabar o tempo.

    Na primeira vez o navegador pede permissão para usar o microfone: clique em Permitir.
    """
    import base64
    from google.colab.output import eval_js
    from IPython.display import Javascript, display

    display(Javascript(_JS_GRAVAR))
    dados = eval_js(f"gravarAnotacao({int(max_segundos * 1000)})", timeout_sec=max_segundos + 60)
    arquivo = Path(arquivo)
    arquivo.write_bytes(base64.b64decode(dados))
    print(f"Gravação recebida: {arquivo.stat().st_size / 1024:.0f} KB")
    if ouvir:   # só para teste: toca a gravação antes de transcrever (o arquivo é apagado depois)
        from IPython.display import Audio
        display(Audio(str(arquivo)))
    return arquivo

## 4. Organização: regras (sempre) + Claude (opcional, só recebe o texto já filtrado)
Além das marcações, a anotação vira um **resumo organizado**: situação relatada, prioridade,
próximos passos e pendências. O resumo aparece só na tela, para o agente revisar.
O que fica salvo continua sendo só as marcações confirmadas.

In [ ]:
@dataclass
class Marcacao:
    categoria: str
    origem: str                 # "regra", "ia" ou "regra+ia"
    evidencia: str = ""         # trecho do texto FILTRADO, só para o agente conferir na tela
    tem_laudo: str | None = None  # só para pessoa_com_deficiencia: "sim", "nao" ou None
    detalhe: str | None = None    # só para melhor_horario: "manhã", "tarde" ou "noite"
    confirmada: bool = False    # o agente confirma no app

    @property
    def rotulo(self) -> str:
        return CATEGORIAS[self.categoria]["rotulo"]


@dataclass
class Resumo:
    situacao: list[str] = field(default_factory=list)        # o que foi relatado, em frases curtas
    prioridade: str = "normal"                                # "alta" quando há urgência social
    proximos_passos: list[str] = field(default_factory=list)
    pendencias: list[str] = field(default_factory=list)      # o que falta saber ou providenciar
    sugestoes: list[str] = field(default_factory=list)       # ações concretas para o agente decidir


@dataclass
class ResultadoAnotacao:
    texto_filtrado: str
    dados_clinicos: list[str]
    marcacoes: list[Marcacao] = field(default_factory=list)
    modo: str = "regras"
    resumo: Resumo = field(default_factory=Resumo)

    @property
    def aviso_esus(self) -> str | None:
        if not self.dados_clinicos:
            return None
        return ("A anotação tinha " + ", ".join(self.dados_clinicos) +
                ". Isso não fica no app: registre no e-SUS.")

    def para_formulario(self, so_confirmadas: bool = False) -> dict:
        """Respostas do formulário que as marcações preenchem (ex.: pcd='sim'), para o simulador."""
        resp = {}
        for m in self.marcacoes:
            if m.confirmada or not so_confirmadas:
                resp.update(CATEGORIAS[m.categoria]["formulario"])
        return resp

    def para_salvar(self, id_caso: str, data: str) -> list[dict]:
        """Linhas de fato_necessidade_caso: só categorias confirmadas. Nada de texto, nada de dado clínico."""
        return [{"id_caso": id_caso, "data": data, "categoria": m.categoria,
                 "tem_laudo": m.tem_laudo, "detalhe": m.detalhe, "origem": m.origem}
                for m in self.marcacoes if m.confirmada]

    def mostrar(self) -> None:
        r = self.resumo
        print(f"[modo: {self.modo}]")
        print("Texto filtrado (só na tela, não é salvo):", self.texto_filtrado)
        if self.aviso_esus:
            print("⚠️ ", self.aviso_esus)
        print("\n📋 RESUMO DA VISITA" + ("   🔴 PRIORIDADE ALTA" if r.prioridade == "alta" else ""))
        if r.situacao:
            print("  Situação relatada:")
            for s in r.situacao:
                print(f"    • {s}")
        if r.proximos_passos:
            print("  Próximos passos:")
            for i, p in enumerate(r.proximos_passos, 1):
                print(f"    {i}. {p}")
        if r.pendencias:
            print("  Pendências (perguntar ou providenciar):")
            for p in r.pendencias:
                print(f"    • {p}")
        if r.sugestoes:
            print("\n💡 SUGESTÕES" + (" DA IA" if "IA" in self.modo else "") + " (o agente decide)")
            for p in r.sugestoes:
                print(f"    → {p}")
        print("\n✅ MARCAÇÕES PARA CONFIRMAR (só elas ficam salvas)")
        if not self.marcacoes:
            print("  Nenhuma marcação sugerida.")
        for m in self.marcacoes:
            extra = (f" · laudo: {m.tem_laudo}" if m.tem_laudo else "") + (f": {m.detalhe}" if m.detalhe else "")
            print(f"  ☐ {m.rotulo}{extra}  [{m.origem}]  → {CATEGORIAS[m.categoria]['acao']}")
            if m.evidencia:
                print(f"      \"{m.evidencia}\"")


def _trecho(texto: str, ini: int, fim: int, folga: int = 30) -> str:
    a, b = max(0, ini - folga), min(len(texto), fim + folga)
    return ("…" if a else "") + texto[a:b].strip() + ("…" if b < len(texto) else "")


def por_regras(texto_filtrado: str) -> list[Marcacao]:
    s = _simples(texto_filtrado)   # mesmo comprimento do original na prática (NFKD sem acento)
    achadas: dict[str, Marcacao] = {}
    for cat, padroes in PADROES.items():
        for p in padroes:
            for m in re.finditer(p, s):
                if NEGACAO.search(s[max(0, m.start() - 25):m.start()]):
                    continue   # "não tem deficiência", "nem precisa de remédio"
                if cat not in achadas:
                    achadas[cat] = Marcacao(cat, "regra", _trecho(texto_filtrado, m.start(), m.end()))
                break
    if "melhor_horario" in achadas:
        per = PERIODO.findall(s)
        achadas["melhor_horario"].detalhe = NOME_PERIODO[per[0]] if per else None
    if "pessoa_com_deficiencia" in achadas:
        achadas["pessoa_com_deficiencia"].tem_laudo = ("nao" if LAUDO_NAO.search(s)
                                                       else "sim" if LAUDO_SIM.search(s) else None)
    return list(achadas.values())


def _frases(texto_filtrado: str) -> list[str]:
    """Quebra o texto em frases e corta cada uma no primeiro dado de saúde removido."""
    saida = []
    for f in re.split(r"(?<=[.!?;])\s+", texto_filtrado):
        f = f.split(MARCADOR)[0]
        f = re.sub(r"[\s,;:.!?]+(e|toma|tem|com)?\s*$", "", f, flags=re.IGNORECASE).strip(" ,;:.!?")
        if len(f.split()) >= 3:
            saida.append(f[0].upper() + f[1:])
    return saida


def resumo_por_regras(texto_filtrado: str, marcacoes: list[Marcacao], clinicos: list[str]) -> Resumo:
    cats = {m.categoria for m in marcacoes}
    ordem = ["urgencia_social", "precisa_atendimento_medico", "dificuldade_locomocao",
             "dificuldade_ir_ao_posto", "precisa_acompanhante", "melhor_horario",
             "problema_beneficio", "pessoa_com_deficiencia",
             "uso_continuo_remedio_ou_fralda", "problema_documentos"]   # o mais urgente primeiro
    passos = [CATEGORIAS[c]["acao"] for c in ordem if c in cats]
    if clinicos:
        passos.append("Registrar no e-SUS: " + ", ".join(clinicos))
    pend = []
    for m in marcacoes:
        if m.categoria == "pessoa_com_deficiencia":
            if m.tem_laudo == "nao":
                pend.append("Providenciar laudo da deficiência (ainda não tem)")
            elif m.tem_laudo is None:
                pend.append("Perguntar se a pessoa com deficiência tem laudo")
    simples = _simples(texto_filtrado)
    acomp = ACOMPANHANTE.search(simples)
    quem = texto_filtrado[acomp.start(3):acomp.end(3)].lower() if acomp else None
    if cats & {"dificuldade_locomocao", "dificuldade_ir_ao_posto", "precisa_acompanhante"} and not quem:
        pend.append("Combinar quem acompanha ou se o atendimento será em casa")
    if "problema_beneficio" in cats:
        pend.append("Perguntar qual benefício, desde quando e se recebeu carta ou aviso")
    if "problema_documentos" in cats:
        pend.append("Ver quais documentos faltam (com a certidão de nascimento dá para tirar os outros)")
    sug = sugestoes_por_regras(texto_filtrado, marcacoes, quem)
    return Resumo(_frases(texto_filtrado), "alta" if "urgencia_social" in cats else "normal", passos, pend, sug)


def sugestoes_por_regras(texto_filtrado: str, marcacoes: list[Marcacao], quem: str | None = None) -> list[str]:
    """Ações concretas montadas a partir do que foi dito (compromisso, horário, acompanhante)."""
    s = _simples(texto_filtrado)
    cats = {m.categoria: m for m in marcacoes}
    pessoa = PESSOA.search(texto_filtrado)
    pessoa = ("a " + pessoa.group(0)) if pessoa and pessoa.group(1).lower() in ("dona", "sra") else \
             ("o " + pessoa.group(0)) if pessoa else "a pessoa atendida"
    local = LOCAL.search(s)
    local = LOCAIS[local.group(1)] if local else None
    data = DATA.search(s)
    data = texto_filtrado[data.start():data.end()] if data else None
    tem_compromisso = bool(local and COMPROMISSO.search(s))
    quando = f" ({data})" if data else ""
    periodo = cats["melhor_horario"].detalhe if "melhor_horario" in cats else None
    sug = []
    if tem_compromisso and periodo:
        sug.append(f"Remarcar o {local}{quando} para a {periodo}, se o horário atual não for nesse período")
    if tem_compromisso and quem:
        sug.append(f"Confirmar com o {quem} a ida ao {local}{quando}" + (f" à {periodo}" if periodo else ""))
    elif tem_compromisso and "precisa_acompanhante" in cats:
        sug.append(f"Definir quem acompanha {pessoa} no {local}{quando}")
    if tem_compromisso and "dificuldade_locomocao" in cats:
        sug.append(f"Pedir atendimento prioritário no {local} (dificuldade de locomoção)")
    if "problema_documentos" in cats and not (tem_compromisso and local == "Poupatempo"):
        sug.append("Agendar o Poupatempo para tirar os documentos que faltam")
    if "problema_beneficio" in cats and not (tem_compromisso and local == "CRAS"):
        sug.append("Agendar atendimento no CRAS para ver o benefício")
    if sug:
        sug.append(f"Avisar {pessoa} pelo WhatsApp sobre o que foi combinado")
    return sug


FERRAMENTA = {
    "name": "registrar_anotacao",
    "description": "Registra as marcações de direitos e o resumo organizado da anotação do agente.",
    "input_schema": {
        "type": "object",
        "properties": {
            "marcacoes": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "categoria": {"type": "string", "enum": list(CATEGORIAS)},
                        "evidencia": {"type": "string", "description": "trecho curto e literal da anotação"},
                        "tem_laudo": {"type": "string", "enum": ["sim", "nao", "nao_informado"]},
                    },
                    "required": ["categoria", "evidencia"],
                },
            },
            "situacao": {"type": "array", "items": {"type": "string"},
                         "description": "o que foi relatado, em 1 a 6 frases curtas e objetivas, na 3ª pessoa"},
            "pendencias": {"type": "array", "items": {"type": "string"},
                           "description": "o que o agente ainda precisa perguntar ou providenciar"},
            "proximos_passos": {"type": "array", "items": {"type": "string"},
                                "description": "passos práticos de orientação social, sem conduta médica"},
            "sugestoes": {"type": "array", "items": {"type": "string"},
                          "description": "ações concretas e específicas para este caso (ex.: remarcar um "
                                         "atendimento para o horário em que o acompanhante pode, avisar a "
                                         "pessoa pelo WhatsApp). Só com base no que foi dito."},
            "melhor_horario": {"type": "string", "enum": ["manhã", "tarde", "noite", "nao_informado"]},
        },
        "required": ["marcacoes", "situacao", "pendencias", "proximos_passos", "sugestoes"],
    },
}
INSTRUCAO = (
    "Você ajuda agentes de saúde e assistentes sociais de São Paulo a organizar anotações de visita. "
    "Marque SOMENTE as categorias que a anotação afirma de forma explícita; na dúvida, não marque. "
    "Não deduza diagnóstico, doença ou condição de saúde. Ignore os trechos '" + MARCADOR + "'. "
    "Categorias: " + "; ".join(f"{k} = {v['rotulo']}" for k, v in CATEGORIAS.items()) + ". "
    "A evidência deve ser um trecho literal e curto da anotação. "
    "No resumo: frases curtas, sem inventar nada que não foi dito, sem nome de doença, remédio ou "
    "medida de saúde, e sem orientação médica. A transcrição pode ter erros de fala: corrija só o óbvio."
)


def _limpo(frases) -> list[str]:
    """Segurança extra: tudo que vem da IA passa de novo pelo filtro clínico."""
    out = []
    for f in frases or []:
        f, _ = filtrar_clinico(str(f))
        f = re.sub(r"[\s,;:.!?]+(e|toma|tem|com)?\s*$", "", f.split(MARCADOR)[0], flags=re.IGNORECASE).strip(" ,;:.!?")
        if len(f.split()) >= 2:
            out.append(f[:200])
    return out


def por_ia(texto_filtrado: str, modelo: str | None = None) -> tuple[list[Marcacao], dict] | None:
    """Usa o Claude se houver ANTHROPIC_API_KEY. Recebe só o texto filtrado. None = sem IA disponível."""
    if not os.getenv("ANTHROPIC_API_KEY"):
        return None
    try:
        import anthropic
    except ImportError:
        return None
    try:
        resp = anthropic.Anthropic().messages.create(
            model=modelo or os.getenv("MODELO_ANOTACOES", "claude-haiku-5-5"),
            max_tokens=1200,
            system=INSTRUCAO,
            tools=[FERRAMENTA],
            tool_choice={"type": "tool", "name": FERRAMENTA["name"]},
            messages=[{"role": "user", "content": f"Anotação do agente:\n\"\"\"{texto_filtrado}\"\"\""}],
        )
    except Exception as e:     # sem internet, chave errada, limite: cai para o modo regras
        print(f"[IA indisponível: {type(e).__name__}; usando só as regras]")
        return None
    bloco = next(b for b in resp.content if b.type == "tool_use")
    out = []
    for m in bloco.input.get("marcacoes", []):
        if m.get("categoria") not in CATEGORIAS:
            continue
        evid, _ = filtrar_clinico(m.get("evidencia", ""))
        laudo = m.get("tem_laudo")
        out.append(Marcacao(m["categoria"], "ia", evid[:160],
                            laudo if laudo in ("sim", "nao") and m["categoria"] == "pessoa_com_deficiencia" else None))
    extras = {k: _limpo(bloco.input.get(k)) for k in ("situacao", "pendencias", "proximos_passos", "sugestoes")}
    for m in out:
        if m.categoria == "melhor_horario" and bloco.input.get("melhor_horario") in ("manhã", "tarde", "noite"):
            m.detalhe = bloco.input["melhor_horario"]
    return out, extras


def _sem_repetir(*listas) -> list[str]:
    vistos, out = set(), []
    for lista in listas:
        for x in lista:
            k = _simples(x).strip(" .")
            if k not in vistos:
                vistos.add(k)
                out.append(x)
    return out


def organizar(texto: str, usar_ia: bool = True) -> ResultadoAnotacao:
    """Texto da anotação (digitado ou transcrito) -> resumo + marcações para o agente confirmar."""
    filtrado, clinicos = filtrar_clinico(texto)
    regras = por_regras(filtrado)
    for tipo, cat in CLINICO_SUGERE.items():
        if tipo in clinicos and all(m.categoria != cat for m in regras):
            regras.append(Marcacao(cat, "regra (remédio citado, nome não guardado)"))
    ordem = list(CATEGORIAS)
    ia = por_ia(filtrado) if usar_ia else None
    if ia is None:
        regras.sort(key=lambda m: ordem.index(m.categoria))
        return ResultadoAnotacao(filtrado, clinicos, regras, "regras (offline)",
                                 resumo_por_regras(filtrado, regras, clinicos))
    marc_ia, extras = ia
    final = {m.categoria: m for m in regras}
    for m in marc_ia:                 # união: a regra garante o básico, a IA pega o que foi dito de outro jeito
        if m.categoria in final:
            final[m.categoria].origem = "regra+ia"
            final[m.categoria].tem_laudo = final[m.categoria].tem_laudo or m.tem_laudo
            final[m.categoria].detalhe = final[m.categoria].detalhe or m.detalhe
        else:
            final[m.categoria] = m
    marcacoes = sorted(final.values(), key=lambda m: ordem.index(m.categoria))
    base = resumo_por_regras(filtrado, marcacoes, clinicos)   # passos fixos de cada marcação vêm sempre
    resumo = Resumo(extras["situacao"] or base.situacao, base.prioridade,
                    _sem_repetir(base.proximos_passos, extras["proximos_passos"]),
                    _sem_repetir(base.pendencias, extras["pendencias"]),
                    _sem_repetir(base.sugestoes, extras["sugestoes"]))
    return ResultadoAnotacao(filtrado, clinicos, marcacoes, "regras + IA (Claude)", resumo)


def anotar_por_voz(caminho_audio: str | Path, usar_ia: bool = True) -> ResultadoAnotacao:
    """Atalho para um arquivo de áudio: transcreve, apaga o áudio e organiza."""
    return organizar(transcrever(caminho_audio), usar_ia=usar_ia)


def anotar_pelo_microfone(max_segundos: int = 180, usar_ia: bool = True,
                          mostrar_transcricao: bool = False, ouvir: bool = False) -> ResultadoAnotacao:
    """Botão de microfone: grava, transcreve, apaga o áudio, filtra e organiza."""
    arquivo = gravar_do_microfone(max_segundos, ouvir=ouvir)
    print("Transcrevendo… (na primeira vez o modelo é baixado, alguns minutos)")
    texto = transcrever(arquivo)
    if mostrar_transcricao:
        print("Transcrição (só na tela):", texto, "\n")
    return organizar(texto, usar_ia=usar_ia)


ARQUIVO_SALVO = Path("fato_necessidade_caso.csv")


def confirmar_e_salvar(res: ResultadoAnotacao, id_caso: str, data: str | None = None,
                       arquivo: str | Path = ARQUIVO_SALVO) -> None:
    """Mostra uma caixa por marcação; o agente marca as que confirma e clica em Salvar.

    Só as confirmadas vão para o arquivo (fato_necessidade_caso). Nada de texto, nome ou dado clínico.
    """
    import datetime
    import pandas as pd
    from IPython.display import display
    data = data or datetime.date.today().isoformat()
    arquivo = Path(arquivo)

    def salvar():
        linhas = res.para_salvar(id_caso, data)
        if not linhas:
            print("Nenhuma marcação confirmada: nada foi salvo.")
            return
        novo = pd.DataFrame(linhas)
        novo.to_csv(arquivo, mode="a", header=not arquivo.exists(), index=False)
        print(f"✅ {len(linhas)} marcação(ões) salvas em {arquivo}:")
        display(novo)

    if not res.marcacoes:
        print("Nenhuma marcação para confirmar.")
        return
    try:
        import ipywidgets as w
    except ImportError:            # sem widgets: confirma tudo (modo demonstração)
        for m in res.marcacoes:
            m.confirmada = True
        salvar()
        return
    caixas = []
    for m in res.marcacoes:
        extra = f": {m.detalhe}" if m.detalhe else (f" · laudo: {m.tem_laudo}" if m.tem_laudo else "")
        caixas.append(w.Checkbox(value=False, description=m.rotulo + extra, indent=False,
                                 layout=w.Layout(width="auto")))
    botao = w.Button(description="Salvar confirmadas", button_style="success", icon="check")
    saida = w.Output()

    def ao_clicar(_):
        for m, c in zip(res.marcacoes, caixas):
            m.confirmada = c.value
        botao.disabled = True
        with saida:
            salvar()

    botao.on_click(ao_clicar)
    display(w.VBox([w.HTML("<b>O agente confirma as marcações (só elas ficam salvas):</b>"), *caixas, botao, saida]))

## Chave do Claude (opcional)

In [ ]:
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('Chave encontrada: modo regras + IA')
except Exception:
    print('Sem chave: modo só regras (offline)')

Sem chave: modo só regras (offline)


## Teste 1 — anotação digitada

In [ ]:
nota = ('Visitei a Dona Cida. Ela está acamada desde o mês passado, pressão 16 por 10, toma losartana. '
        'O neto tem autismo, ainda não tem laudo. Disse que está sem comida em casa.')
res = organizar(nota)
res.mostrar()

[modo: regras (offline)]
Texto filtrado (só na tela, não é salvo): Visitei a Dona Cida. Ela está acamada desde o mês passado, [dado de saúde: registrar no e-SUS] toma [dado de saúde: registrar no e-SUS] . O neto tem autismo, ainda não tem laudo. Disse que está sem comida em casa.
⚠️  A anotação tinha pressão, nome de remédio. Isso não fica no app: registre no e-SUS.

📋 RESUMO DA VISITA   🔴 PRIORIDADE ALTA
  Situação relatada:
    • Visitei a Dona Cida
    • Ela está acamada desde o mês passado
    • O neto tem autismo, ainda não tem laudo
    • Disse que está sem comida em casa
  Próximos passos:
    1. Prioridade no caso e aviso ao coordenador
    2. Atendimento em casa ou acompanhado
    3. Conferir BPC; incluir laudo na lista do que levar
    4. Conferir Farmácia Popular
    5. Registrar no e-SUS: pressão, nome de remédio
  Pendências (perguntar ou providenciar):
    • Providenciar laudo da deficiência (ainda não tem)
    • Combinar quem acompanha ou se o atendimento será em casa

✅

## O agente confirma → o que vai para o formulário e o que fica salvo

In [ ]:
import pandas as pd
for m in res.marcacoes:          # no app, o agente marca uma por uma
    m.confirmada = True
print('Preenche no formulário do simulador:', res.para_formulario(so_confirmadas=True))
print('\nLinhas de fato_necessidade_caso (único dado salvo):')
display(pd.DataFrame(res.para_salvar('3729', '2026-11-14')))

Preenche no formulário do simulador: {'pcd': 'sim', 'precisa_remedio_ou_fralda': 'sim'}

Linhas de fato_necessidade_caso (único dado salvo):


,id_caso,data,categoria,tem_laudo,detalhe,origem
0,3729,2026-11-14,dificuldade_locomocao,None,None,regra
1,3729,2026-11-14,pessoa_com_deficiencia,nao,None,regra
2,3729,2026-11-14,uso_continuo_remedio_ou_fralda,None,None,"regra (remédio citado, nome não guardado)"
3,3729,2026-11-14,urgencia_social,None,None,regra


## Teste 2 — outros exemplos (inclusive fala solta, sem pontuação)

In [ ]:
exemplos = [
    'Ela não tem deficiência e não precisa de remédio. Mora longe do CRAS e está sem dinheiro para passagem.',
    'Seu Antônio é cadeirante, tem laudo. Pesa 82 kg, mede 1,70 m, glicemia 210. Tem diabetes.',
    'então hoje eu passei na casa da dona maria ela é cega e mora sozinha e a luz foi cortada semana passada',
    'O Bolsa Família dela foi bloqueado mês passado e o cadastro está desatualizado.',
    'o filho perdeu o RG e não tem certidão, ela anda de bengala e não consegue subir escada',
    'Dona Cida tem horário marcado no Poupatempo dia 18 de novembro. Ela tem dificuldade para andar, '
    'então precisa ir com o filho, que só está disponível à tarde.',
    'Tudo certo com a família, nada a registrar.',
]
for e in exemplos:
    print('\n>>>', e)
    organizar(e).mostrar()


>>> Ela não tem deficiência e não precisa de remédio. Mora longe do CRAS e está sem dinheiro para passagem.
[modo: regras (offline)]
Texto filtrado (só na tela, não é salvo): Ela não tem deficiência e não precisa de remédio. Mora longe do CRAS e está sem dinheiro para passagem.

📋 RESUMO DA VISITA
  Situação relatada:
    • Ela não tem deficiência e não precisa de remédio
    • Mora longe do CRAS e está sem dinheiro para passagem
  Próximos passos:
    1. Encaminhamento por visita domiciliar ou atendimento acompanhado
  Pendências (perguntar ou providenciar):
    • Combinar quem acompanha ou se o atendimento será em casa

✅ MARCAÇÕES PARA CONFIRMAR (só elas ficam salvas)
  ☐ Dificuldade de chegar ao posto (distância, passagem, sem acompanhante)  [regra]  → Encaminhamento por visita domiciliar ou atendimento acompanhado
      "…cia e não precisa de remédio. Mora longe do CRAS e está sem dinheiro p…"

>>> Seu Antônio é cadeirante, tem laudo. Pesa 82 kg, mede 1,70 m, glicemia 210. Tem di

## Teste 3 — fale a anotação pelo microfone
Rode a célula, clique em **Permitir** se o navegador pedir o microfone e fale como se estivesse terminando a visita, por exemplo:

> *"Visitei o seu José, ele mora sozinho, usa cadeira de rodas e não tem quem leve ele no posto. Toma remédio todo dia. A geladeira estava vazia."*

Clique em **Parar e organizar** quando terminar (ou a gravação para sozinha em 3 minutos). Na primeira vez o modelo de transcrição é baixado (alguns minutos). O áudio é apagado logo depois da transcrição.

In [ ]:
USAR_MICROFONE = True
if USAR_MICROFONE:
    res_voz = anotar_pelo_microfone(max_segundos=180, mostrar_transcricao=True, ouvir=True)
    res_voz.mostrar()
    confirmar_e_salvar(res_voz, id_caso="3729")

<IPython.core.display.Javascript object>

Gravação recebida: 172 KB


Transcrevendo… (na primeira vez o modelo é baixado, alguns minutos)
Áudio recebido: 12.0 s · fala detectada: 11.6 s
Transcrição (só na tela): Dona Cida tem horário marcado no Poupatempo, no dia 18 de novembro. Ela tem dificuldade para andar, então precisa ir com o filho, só estar disponível à tarde. 

[modo: regras (offline)]
Texto filtrado (só na tela, não é salvo): Dona Cida tem horário marcado no Poupatempo, no dia 18 de novembro. Ela tem dificuldade para andar, então precisa ir com o filho, só estar disponível à tarde.

📋 RESUMO DA VISITA
  Situação relatada:
    • Dona Cida tem horário marcado no Poupatempo, no dia 18 de novembro
    • Ela tem dificuldade para andar, então precisa ir com o filho, só estar disponível à tarde
  Próximos passos:
    1. Atendimento em casa ou acompanhado
    2. Confirmar quem acompanha e se a pessoa está livre no dia e horário
    3. Marcar atendimentos neste período

💡 SUGESTÕES (o agente decide)
    → Remarcar o Poupatempo (dia 18 de novembro) para 

## Teste 4 — enviar um áudio gravado no celular (opcional)
Grave pelo próprio WhatsApp ou gravador, mude `USAR_ARQUIVO` para `True` e envie o arquivo quando pedir.

In [ ]:
USAR_ARQUIVO = True
if USAR_ARQUIVO:
    from google.colab import files
    enviado = files.upload()
    res_arq = anotar_por_voz(next(iter(enviado)))     # transcreve, apaga o arquivo e organiza
    res_arq.mostrar()
    confirmar_e_salvar(res_arq, id_caso="3729")

Saving Voz 261114_192737.m4a to Voz 261114_192737.m4a
Áudio recebido: 12.0 s · fala detectada: 11.4 s
[modo: regras (offline)]
Texto filtrado (só na tela, não é salvo): Dona Acida tem horário marcado no Poupatempo dia 18 de novembro. Ela tem dificuldade para andar, então precisa ir com o filho, que só está disponível à tarde.

📋 RESUMO DA VISITA
  Situação relatada:
    • Dona Acida tem horário marcado no Poupatempo dia 18 de novembro
    • Ela tem dificuldade para andar, então precisa ir com o filho, que só está disponível à tarde
  Próximos passos:
    1. Atendimento em casa ou acompanhado
    2. Confirmar quem acompanha e se a pessoa está livre no dia e horário
    3. Marcar atendimentos neste período

💡 SUGESTÕES (o agente decide)
    → Remarcar o Poupatempo (dia 18 de novembro) para a tarde, se o horário atual não for nesse período
    → Confirmar com o filho a ida ao Poupatempo (dia 18 de novembro) à tarde
    → Pedir atendimento prioritário no Poupatempo (dificuldade de locomoçã